# Caso 01 — BrightCart (Moda Online): Exploración Inicial y Auditoría de Calidad
## Día 1 — Tu primer encargo y los datos

En este primer cuaderno exploramos los **7 ficheros CSV** exportados de producción tras el cierre de la campaña de **Rebajas de Verano 2024** (`lineas_pedido.csv`, `pedidos.csv`, `productos.csv`, `clientes.csv`, `campana.csv`, `devoluciones.csv` y `visitas_web.csv`), aplicando la regla de oro de **Diego (Head de Datos)**: *«Nunca calcular antes de explorar y entender qué representa cada fila»*.

### Objetivos del Día 1:
1. Verificar el inventario de tablas, dimensiones (`.shape`), tipos de datos (`dtypes`) y valores nulos (`.isnull().sum()`).
2. Distinguir la **Tabla de Hechos** (`lineas_pedido.csv`, granularidad de 1 línea de prenda vendida) de las **Tablas de Dimensión y Contexto** (`pedidos`, `productos`, `clientes`, `campana`, `devoluciones`, `visitas_web`).
3. Auditar la unicidad de claves primarias (`len()` vs `.nunique()`) y cuantificar las anomalías de exportación (duplicados, precios como texto con `€`, descuentos en tres formatos, fechas mixtas y devoluciones huérfanas).

In [1]:
# Verificación de entorno e inventario general de los 7 ficheros CSV
import os
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path('data/raw') if Path('data/raw').exists() else Path('data')
archivos = sorted([f.name for f in DATA_DIR.glob('*.csv')])
print('Archivos CSV disponibles:', archivos)

resumen_inventario = []
for archivo in archivos:
    df_tmp = pd.read_csv(DATA_DIR / archivo)
    pk_col = df_tmp.columns[0]
    resumen_inventario.append({
        'fichero': archivo,
        'filas': len(df_tmp),
        'columnas': df_tmp.shape[1],
        'col_clave': pk_col,
        'unicos_clave': df_tmp[pk_col].nunique(),
        'nulos_totales': int(df_tmp.isnull().sum().sum()),
        'lista_columnas': ', '.join(df_tmp.columns)
    })

df_inventario = pd.DataFrame(resumen_inventario)
display(df_inventario)

Archivos CSV disponibles: ['campana.csv', 'clientes.csv', 'devoluciones.csv', 'lineas_pedido.csv', 'pedidos.csv', 'productos.csv', 'visitas_web.csv']


,fichero,filas,columnas,col_clave,unicos_clave,nulos_totales,lista_columnas
0,campana.csv,4,8,campana_id,4,0,"campana_id, nombre, tipo, fecha_inicio, fecha_..."
1,clientes.csv,5010,5,cliente_id,4970,150,"cliente_id, fecha_registro, ciudad, genero, edad"
2,devoluciones.csv,5558,5,devolucion_id,5558,0,"devolucion_id, linea_id, fecha_devolucion, tal..."
3,lineas_pedido.csv,20140,7,linea_id,20000,0,"linea_id, pedido_id, producto_id, talla, canti..."
4,pedidos.csv,8000,7,pedido_id,8000,1179,"pedido_id, cliente_id, fecha_pedido, campana_i..."
5,productos.csv,400,7,producto_id,400,12,"producto_id, nombre, categoria, temporada, col..."
6,visitas_web.csv,112,3,fecha,28,4,"fecha, canal, sesiones"


### Análisis del Inventario General de Tablas
* El modelo gira alrededor de **`lineas_pedido.csv`** (`20.140` filas y `7` columnas) como **Tabla de Hechos** central: cada registro representa una línea de producto dentro de un carrito de compra. Sin embargo, solo existen **`20.000` `linea_id` únicos**, lo que delata **`140` registros duplicados** en la exportación SQL.
* En las tablas de contexto también aparecen alertas inmediatas:
  * **`clientes.csv`** tiene `5.010` filas para `4.970` `cliente_id` únicos (**`40` fichas duplicadas**) y `150` valores nulos.
  * **`pedidos.csv`** contiene `8.000` pedidos únicos con `1.179` nulos en `campana_id` (compras orgánicas fuera de campaña).
  * **`productos.csv`** (`400` referencias) presenta `12` valores nulos en `coste_unitario`, y **`visitas_web.csv`** (`112` registros diarios por canal) tiene `4` nulos en `sesiones`.

---
## Reto 1: Inspeccionar la tabla `clientes.csv`
Aplicamos la rutina de detective sobre la dimensión de clientes: dimensiones, primeras filas, tipos de datos, conteo de nulos y prueba de unicidad (`len` vs `nunique`).

In [2]:
# =============================================================================
# RETO 1: EXPLORACIÓN DE CLIENTES
# =============================================================================
clientes = pd.read_csv('data/clientes.csv')
print('Dimensiones:', clientes.shape)
display(clientes.head())
print('\nTipos de datos:\n', clientes.dtypes)
print('\nValores nulos:\n', clientes.isnull().sum())
print('\nTotal de filas (len):', len(clientes))
print('Clientes únicos (nunique):', clientes['cliente_id'].nunique())
print('Duplicados de cliente_id:', len(clientes) - clientes['cliente_id'].nunique())
print('Valores únicos de género:', clientes['genero'].unique().tolist())
print('Muestra de ciudades:', sorted(clientes['ciudad'].unique())[:10])

Dimensiones: (5010, 5)


,cliente_id,fecha_registro,ciudad,genero,edad
0,2371,2019-10-18,bilbao,F,29
1,3537,2021-10-14,Bilbao,M,37
2,2088,2023-10-05,madrid,m,36
3,2769,2018-01-01,Lisboa,F,36
4,1567,2021-06-25,Madrid,M,42



Tipos de datos:
 cliente_id        int64
fecha_registro      str
ciudad              str
genero              str
edad              int64
dtype: object

Valores nulos:
 cliente_id          0
fecha_registro    150
ciudad              0
genero              0
edad                0
dtype: int64

Total de filas (len): 5010
Clientes únicos (nunique): 4970
Duplicados de cliente_id: 40
Valores únicos de género: ['F', 'M', 'm', 'Femenino', 'Masculino', 'f']
Muestra de ciudades: ['A CORUÑA', 'A Coruña', 'BARCELONA', 'BILBAO', 'Barcelona', 'Bilbao', 'LISBOA', 'Lisboa', 'MADRID', 'Madrid']


### Análisis del Reto 1 (`clientes.csv`)
* **Duplicidad de clave primaria:** Existen **`5.010` filas** pero solo **`4.970` `cliente_id` únicos** (`40` clientes repetidos que inflarían cualquier conteo directo sin deduplicar).
* **Inconsistencias de formato en texto:** La columna `ciudad` mezcla minúsculas y mayúsculas (`'bilbao'` vs `'Bilbao'`, `'madrid'` vs `'Madrid'`), al igual que `genero` (`'M'` vs `'m'`, `'F'` vs `'f'`).
* **Valores ausentes:** Hay **`150` registros sin `fecha_registro`**, lo que impide calcular la antigüedad de ese `3,0 %` de la base pero no afecta la facturación de la campaña.

---
## Reto 2: Inspeccionar la Tabla de Hechos (`lineas_pedido.csv`)
Esta es la tabla nuclear del negocio: cada fila representa una prenda concreta vendida dentro de un pedido. Revisamos su volumen, tipos de datos y las trampas de formato en precios, descuentos y cantidades.

In [3]:
# =============================================================================
# RETO 2: EXPLORACIÓN DE LÍNEAS DE PEDIDO (TABLA DE HECHOS)
# =============================================================================
lineas = pd.read_csv('data/lineas_pedido.csv')
print('Dimensiones:', lineas.shape)
display(lineas.head())
print('\nTipos de datos:\n', lineas.dtypes)
print('\nTotal filas (len):', len(lineas))
print('Líneas únicas (nunique):', lineas['linea_id'].nunique())
print('\nMuestra de descuentos únicos:', lineas['descuento_pct'].unique()[:10])
print('Muestra de precios únicos:', lineas['precio_unitario'].unique()[:5])
print('Valores únicos de cantidad:', sorted(lineas['cantidad'].unique()))

Dimensiones: (20140, 7)


,linea_id,pedido_id,producto_id,talla,cantidad,precio_unitario,descuento_pct
0,900001,500001,1137,39,1,"63,83 €",0
1,900002,500002,1232,XL,1,"26,48 €",0.3
2,900003,500002,1028,L,2,"48,82 €",30%
3,900004,500003,1229,S,1,"38,72 €",0.0
4,900005,500003,1284,S,2,"20,70 €",0%



Tipos de datos:
 linea_id           int64
pedido_id          int64
producto_id        int64
talla                str
cantidad           int64
precio_unitario      str
descuento_pct        str
dtype: object

Total filas (len): 20140
Líneas únicas (nunique): 20000

Muestra de descuentos únicos: <ArrowStringArray>
['0', '0.3', '30%', '0.0', '0%', '0.2', '20', '40', '0.4', '10%']
Length: 10, dtype: str
Muestra de precios únicos: <ArrowStringArray>
['63,83 €', '26,48 €', '48,82 €', '38,72 €', '20,70 €']
Length: 5, dtype: str
Valores únicos de cantidad: [np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


### Análisis del Reto 2 (`lineas_pedido.csv`)
* **Tres anomalías críticas que impiden sumar directamente:**
  1. **`precio_unitario` como texto (`str`):** Viene formateado con coma decimal española y símbolo de moneda (`'63,83 €'`, `'26,48 €'`).
  2. **`descuento_pct` en tres escalas distintas (`str`):** Mezcla porcentajes con símbolo (`'30%'`, `'10%'`), decimales (`'0.3'`, `'0.2'`) y enteros (`'20'`, `'40'`). Sumar o multiplicar esta columna sin normalizar provocaría errores masivos e invisibles.
  3. **Duplicados y cantidades inválidas:** Hay **`140` `linea_id` duplicados** (`20.140` filas frente a `20.000` IDs únicos) y existen líneas con cantidades `<= 0` (`-1`, `0`) que deben depurarse en el Día 2.

---
## Reto 3: Exploración de `pedidos.csv` y Relación de Granularidad
Comprobamos cómo se relaciona la cabecera de pedidos (`pedidos.csv`) con las líneas de detalle (`lineas_pedido.csv`) y auditamos los formatos de `fecha_pedido` y `campana_id`.

In [4]:
# =============================================================================
# RETO 3: EXPLORACIÓN DE PEDIDOS Y GRANULARIDAD
# =============================================================================
pedidos = pd.read_csv('data/pedidos.csv')
print('Dimensiones de pedidos:', pedidos.shape)
display(pedidos.head())
print('\nPedidos únicos en pedidos.csv:', pedidos['pedido_id'].nunique())
print('Pedidos únicos presentes en lineas_pedido:', lineas['pedido_id'].nunique())
print('Promedio bruto de líneas por pedido:', len(lineas) / len(pedidos))
print('\nNulos por columna en pedidos:\n', pedidos.isnull().sum())
print('\nDistribución de campana_id (incluyendo nulos):')
print(pedidos['campana_id'].value_counts(dropna=False))
print('\nMuestra de fechas en pedidos:\n', pedidos['fecha_pedido'].head())

Dimensiones de pedidos: (8000, 7)


,pedido_id,cliente_id,fecha_pedido,campana_id,canal,ciudad,metodo_pago
0,500001,2290,24/06/2024,NaN,directo,Oporto,tarjeta
1,500002,4937,2024-07-02,1.0,instagram,lisboa,tarjeta
2,500003,544,2024-06-27,NaN,email,Oporto,bizum
3,500004,34,2024-07-12,1.0,email,Zaragoza,paypal
4,500005,4507,2024-07-05,1.0,instagram,Madrid,paypal



Pedidos únicos en pedidos.csv: 8000
Pedidos únicos presentes en lineas_pedido: 8000
Promedio bruto de líneas por pedido: 2.5175

Nulos por columna en pedidos:
 pedido_id          0
cliente_id         0
fecha_pedido       0
campana_id      1179
canal              0
ciudad             0
metodo_pago        0
dtype: int64

Distribución de campana_id (incluyendo nulos):
campana_id
1.0    6821
NaN    1179
Name: count, dtype: int64

Muestra de fechas en pedidos:
 0    24/06/2024
1    2024-07-02
2    2024-06-27
3    2024-07-12
4    2024-07-05
Name: fecha_pedido, dtype: str


### Análisis del Reto 3 (`pedidos.csv`)
* **Integridad referencial:** Los **`8.000` `pedido_id`** de `pedidos.csv` son únicos y coinciden al `100 %` con los `8.000` pedidos presentes en `lineas_pedido.csv` (a razón de ~`2,5` líneas por pedido).
* **Significado de negocio de los nulos en `campana_id`:** Los **`1.179` valores `NaN`** en `campana_id` no son un fallo de calidad, sino las **compras orgánicas realizadas en la semana previa** (`24/06/2024` a `30/06/2024`) antes del arranque de las Rebajas de Verano (`campana_id = 1.0`, que concentra `6.821` pedidos).
* **Fechas mixtas:** La columna `fecha_pedido` combina el formato europeo `DD/MM/YYYY` (`'24/06/2024'`) con el estándar ISO `YYYY-MM-DD` (`'2024-07-02'`).

---
## Reto 4: Inspección de `devoluciones.csv` y Detección de Registros Huérfanos
En moda online, las devoluciones son el principal sumidero de margen. Verificamos cuántas devoluciones existen y si todas apuntan a una línea de venta real en `lineas_pedido.csv`.

In [5]:
# =============================================================================
# RETO 4: DEVOLUCIONES Y REGISTROS HUÉRFANOS
# =============================================================================
devoluciones = pd.read_csv('data/devoluciones.csv')
print('Dimensiones:', devoluciones.shape)
display(devoluciones.head())
print('\nTotal de filas:', len(devoluciones))
print('Devoluciones únicas:', devoluciones['devolucion_id'].nunique())
existe = devoluciones['linea_id'].isin(lineas['linea_id'])
print('\nDevoluciones con línea válida:', int(existe.sum()))
print('Devoluciones huérfanas (sin línea de venta):', int((~existe).sum()))
print('\nMotivos de devolución en crudo:')
print(devoluciones['motivo'].value_counts(dropna=False))

Dimensiones: (5558, 5)


,devolucion_id,linea_id,fecha_devolucion,talla,motivo
0,702838,910363,15/07/2024,36,talla incorrecta
1,700079,900324,18/07/2024,XL,no me gusta
2,704821,917455,2024-07-16,m,talla incorrecta
3,701713,906266,17/07/2024,s,no me gusta
4,701621,905938,2024-07-05,36,talla incorrecta



Total de filas: 5558
Devoluciones únicas: 5558

Devoluciones con línea válida: 5516
Devoluciones huérfanas (sin línea de venta): 42

Motivos de devolución en crudo:
motivo
talla incorrecta     1509
no me gusta          1187
no me queda bien     1143
defecto               699
cambió de opinión     517
color distinto        458
Talla                  27
me queda pequeño       18
Name: count, dtype: int64


### Análisis del Reto 4 (`devoluciones.csv`)
* **Fuga de integridad referencial:** De las **`5.558` devoluciones** registradas, **`42` son devoluciones huérfanas** cuyos `linea_id` no existen en `lineas_pedido.csv` (correspondientes a pedidos antiguos fuera de la ventana exportada). Solo **`5.516` devoluciones** están vinculadas a líneas válidas.
* **concentración de motivos:** Ya en crudo se observa que las causas ligadas al ajuste de la prenda (`'talla incorrecta'`, `'no me queda bien'`, `'Talla'`, `'me queda pequeño'`) dominan abrumadoramente frente a `'no me gusta'` o `'defecto'`.

---
## Reto 5: Catálogo (`productos.csv`), Campañas (`campana.csv`) y Tráfico (`visitas_web.csv`)
Completamos la auditoría del Día 1 inspeccionando las tres tablas restantes de contexto para cerrar el Diccionario de Datos.

In [6]:
# =============================================================================
# RETO 5: CATÁLOGO, CAMPAÑAS Y TRÁFICO WEB
# =============================================================================
productos = pd.read_csv('data/productos.csv')
campana = pd.read_csv('data/campana.csv')
visitas = pd.read_csv('data/visitas_web.csv')

print('Productos shape:', productos.shape)
print('Nulos en productos:\n', productos.isnull().sum())
print('Categorías distintas en crudo:', productos['categoria'].nunique())
print('Muestra de categorías mal escritas:', sorted(productos['categoria'].unique())[:12])

print('\nTabla de Campañas:')
display(campana)

print('\nVisitas shape:', visitas.shape)
print('Nulos en visitas:\n', visitas.isnull().sum())

Productos shape: (400, 7)
Nulos en productos:
 producto_id        0
nombre             0
categoria          0
temporada          0
coleccion          0
coste_unitario    12
precio_base        0
dtype: int64
Categorías distintas en crudo: 35
Muestra de categorías mal escritas: ['ABRIGOS', 'ACCESORIOS', 'Abrigos', 'Accesorios', 'CALZADO', 'CAMISETAS', 'Calzado', 'Camisetas', 'FALDAS', 'Faldas', 'JERSEIS', 'Jerseis']

Tabla de Campañas:


,campana_id,nombre,tipo,fecha_inicio,fecha_fin,canal,presupuesto,ventas_ano_anterior
0,1,Rebajas Verano 2024,rebajas,2024-07-01,2024-07-21,instagram,48000.0,512340.0
1,2,Rebajas Invierno 2024,rebajas,2024-01-07,2024-01-28,email,52000.0,498120.0
2,3,Black Friday 2023,black_friday,2023-11-24,2023-11-27,google,61.000 €,405780.0
3,4,Vuelta al Cole 2023,promo,2023-09-01,2023-09-15,instagram,22000.0,176900.0



Visitas shape: (112, 3)
Nulos en visitas:
 fecha       0
canal       0
sesiones    4
dtype: int64


### Análisis del Reto 5 (`productos`, `campana` y `visitas_web`)
* **`productos.csv` (`400` SKUs):** Presenta **`12` valores nulos en `coste_unitario`** y una grave fragmentación en `categoria`, donde las **8 familias reales de producto aparecen escritas de `35` maneras distintas** (`'Vestidos'`, `'vestido'`, `'VESTIDOS'`, `'vestidos'`, `'Calzado'`, `'Zapatos'`, etc.).
* **`campana.csv` (`4` campañas):** La campaña objeto de nuestro análisis es **`campana_id = 1` (`Rebajas Verano 2024`)**, activa del `2024-07-01` al `2024-07-21`, con canal principal `instagram`, un **presupuesto de `48.000,0 €`** y una referencia de **ventas del año anterior de `512.340,0 €`**. Se detecta además que la campaña 3 tiene el presupuesto como texto (`'61.000 €'`).
* **`visitas_web.csv` (`112` filas):** Registra las sesiones diarias por canal (`4` canales × `28` días entre el `24/06/2024` y el `21/07/2024`), con **`4` valores nulos** en la columna `sesiones`.

---
## Conclusión Ejecutiva del Día 1
1. **Arquitectura validada:** El modelo en estrella tiene a `lineas_pedido.csv` en el centro (granularidad: 1 línea de prenda vendida) rodeada por `pedidos`, `productos`, `clientes`, `campana` y `devoluciones`, más `visitas_web` agregada por fecha y canal.
2. **Por qué no podíamos calcular hoy:** Sumar directamente sobre los CSV crudos habría incluido `140` líneas duplicadas, cantidades negativas/cero, precios en texto, descuentos multiplicados por 100, categorías partidas en 35 etiquetas y `42` devoluciones huérfanas.
3. **Hoja de ruta para el Día 2:** Ejecutar el pipeline de limpieza y estandarización (`data/clean/`) antes de calcular el primer euro de facturación.